In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import joblib

df = pd.read_csv(r"D:\Project\market-trend-nlp\market-trend-nlp\data\Dataset-CNBCI-Sentimented.csv")
print("Data berhasil dimuat!")

Data berhasil dimuat!


In [2]:
def clean_text(text):
    text = str(text).lower() 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = re.sub(r'\@\w+|\#','', text) 
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text

df['cleaned_text'] = df['judul'].apply(clean_text)
print("Teks berhasil dibersihkan!")

Teks berhasil dibersihkan!


In [3]:
X = df['cleaned_text']
y = df['sentimen'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)) 
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Dimensi data training: {X_train_vec.shape}")

Dimensi data training: (7855, 5000)


In [4]:
svm_model = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
print("Melatih model SVM...")
svm_model.fit(X_train_vec, y_train)

y_pred = svm_model.predict(X_test_vec)
print("\n=== Laporan Evaluasi Model ===")
print(classification_report(y_test, y_pred))

print(f"F1-Score : {f1_score(y_test, y_pred, average='weighted'):.2f}")

Melatih model SVM...

=== Laporan Evaluasi Model ===
              precision    recall  f1-score   support

     negatif       0.82      0.75      0.79       515
      netral       0.82      0.87      0.84       871
     positif       0.83      0.81      0.82       578

    accuracy                           0.82      1964
   macro avg       0.82      0.81      0.82      1964
weighted avg       0.82      0.82      0.82      1964

F1-Score : 0.82


In [5]:
model_path = r"D:\Project\market-trend-nlp\market-trend-nlp\models\svm_sentiment_model.pkl"
vectorizer_path = r"D:\Project\market-trend-nlp\market-trend-nlp\models\tfidf_vectorizer.pkl"

joblib.dump(svm_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

print("Model dan Vectorizer berhasil diexport ke folder 'models'!")

Model dan Vectorizer berhasil diexport ke folder 'models'!


In [ ]:
model_path = r"D:\Project\market-trend-nlp\market-trend-nlp\models\svm_sentiment_model.pkl"
vectorizer_path = r"D:\Project\market-trend-nlp\market-trend-nlp\models\tfidf_vectorizer.pkl"

loaded_model = joblib.load(model_path)
loaded_vectorizer = joblib.load(vectorizer_path)

berita_baru = [
    "IHSG anjlok parah setelah pengumuman kenaikan suku bunga The Fed yang agresif.", 
    "Investasi asing masuk triliunan rupiah, ekonomi Indonesia meroket pesat!", 
    "Pemerintah merilis laporan anggaran kuartal ketiga pada hari ini." 
]

berita_bersih = [clean_text(berita) for berita in berita_baru]

fitur_baru = loaded_vectorizer.transform(berita_bersih)
prediksi = loaded_model.predict(fitur_baru)
probabilitas = loaded_model.predict_proba(fitur_baru) 

print("=== Hasil Sanity Check Model ===")
for i in range(len(berita_baru)):
    print(f"\nBerita Asli   : {berita_baru[i]}")
    print(f"Hasil Prediksi: {prediksi[i].upper()}")
    print(f"Confidence    : {max(probabilitas[i]) * 100:.2f}%")

=== Hasil Sanity Check Model ===

Berita Asli   : IHSG anjlok parah setelah pengumuman kenaikan suku bunga The Fed yang agresif.
Hasil Prediksi: NEGATIF
Confidence    : 93.73%

Berita Asli   : Investasi asing masuk triliunan rupiah, ekonomi Indonesia meroket pesat!
Hasil Prediksi: POSITIF
Confidence    : 92.25%

Berita Asli   : Pemerintah merilis laporan anggaran kuartal ketiga pada hari ini.
Hasil Prediksi: NETRAL
Confidence    : 76.61%
